# 01 Piezometer locations

<!-- navigation -->
[<kbd>← 00 Set up project</kbd>](00%20Set%20up%20project.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>10 Import time series from Excel →</kbd>](10%20Import%20time%20series%20from%20Excel.ipynb)

**Required** · minimal path: 00 → **01** → 10 → 20 → 51 → 52

Reads and checks the piezometer table, and writes the piezometers to the project geopackage, with a map per transect.

| | |
|---|---|
| **Input** | `piezometers.xlsx` (template), cross-section line in the transects metadata (for the map) |
| **Output** | `project_data.gpkg`, layer `piezometers` |

One row per piezometer filter with its transect, coordinates, filter screen, model layer and `x_crosssection`: the x-coordinate in the model, i.e. the distance from the boundary between foreland and hinterland (usually the inner toe of the levee), positive towards the hinterland. You determine this distance yourself, e.g. in a GIS.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt
print(f"timflow_levee_toolkit {tlt.__version__}")

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

## 1. Read the piezometers

In [ ]:
piezometers = tlt.read_piezometer_table(project.general_file("piezometers"))
piezometers

## 2. x-coordinate and model layer
Check the distances and model layers per transect (the map below shows them as well).

In [ ]:
piezometers.sort_values(["transect", "x_crosssection", "modellayer"])[
    ["name", "transect", "x_crosssection", "modellayer", "screen_top", "screen_bottom", "use_calibration"]
]

## 3. Save to the project geopackage

In [ ]:
gdf = tlt.write_piezometers(
    piezometers,
    project.general_file("project_data"),
    layer=project.settings["paths"]["piezometer_layer"],
    crs=project.crs,
)
print(f"{len(gdf)} piezometers written to {project.relpath(project.general_file('project_data'))}")

## 4. Map per transect
Cross-section line (from the river side) and the piezometers with their `x_crosssection`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from adjustText import adjust_text

metadata = project.metadata().set_index("transect")
for transect, group in gdf.groupby("transect"):
    m = metadata.loc[transect]
    f, ax = plt.subplots(figsize=(7, 7))
    if pd.notna(m.get("x1_xsec")):
        ax.plot([m["x1_xsec"], m["x2_xsec"]], [m["y1_xsec"], m["y2_xsec"]], "k-", label="cross-section")
        ax.plot(m["x1_xsec"], m["y1_xsec"], "bs", label="start (river side)")
    ax.scatter(group["x"], group["y"], c="C1", zorder=5, label="piezometers")
    # One label per well (filters of the same well have the same location)
    wells = group.drop_duplicates(["x", "y"])
    texts = [ax.text(row["x"], row["y"], f"{row['name'].rsplit('_', 1)[0]}\nx = {row['x_crosssection']:.0f} m",
                     fontsize=7, zorder=6, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))
             for _, row in wells.iterrows()]
    line_x = [m["x1_xsec"], m["x2_xsec"]] if pd.notna(m.get("x1_xsec")) else []
    line_y = [m["y1_xsec"], m["y2_xsec"]] if pd.notna(m.get("x1_xsec")) else []
    tlt.square_extent(ax, [*group["x"], *line_x], [*group["y"], *line_y])  # with some surroundings
    tlt.add_basemap(ax, crs=project.crs)  # OpenStreetMap; source="brt" for the topographic map of PDOK
    ax.ticklabel_format(useOffset=False, style="plain")
    # Move the labels apart where they overlap (after the extent is set), with a line to the point
    adjust_text(texts, x=list(wells["x"]), y=list(wells["y"]), ax=ax, expand=(1.2, 1.4),
                arrowprops=dict(arrowstyle="-", color="0.3", lw=0.5))
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.legend(loc=(0, 1), frameon=False, ncol=2, fontsize="x-small")
    tlt.finalize_and_save(f, project.transect_dir(transect, "intermediate"), f"{project.name} - {transect}", tag="piezometers_map", file_label=transect)

---

<!-- navigation -->
[<kbd>← 00 Set up project</kbd>](00%20Set%20up%20project.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>10 Import time series from Excel →</kbd>](10%20Import%20time%20series%20from%20Excel.ipynb)